In [ ]:
import sys
import os

sys.path.append(os.path.abspath('scripts'))

import configparser
from pypcd4 import PointCloud
from bev_detector import BEV_DETECTOR
from localizer import Localizer
from ultralytics import YOLO
import json
import time
import matplotlib.pyplot as plt
import numpy as np

config = configparser.ConfigParser()
config.read('scripts/settings.conf')

In [ ]:
model = YOLO('scripts/model/model.engine', task='obb')
detector = BEV_DETECTOR(model)

fold_nums = sorted(os.listdir("training/PCD"))

err1 = []
err2 = []

for fold_num in fold_nums:
    pcd_nums = ["00"]
    pcd_nums = pcd_nums + [f"{i:02d}" for i in range(80)]

    hd_map_path = f"training/HD_my/{fold_num}.pcd"
    poses_folder_path = f"training/PCD/{fold_num}/poses.json"

    hd_map = PointCloud.from_path(hd_map_path)
    hd_map = hd_map.numpy(("x", "y", "z"))

    scan_localizer = Localizer(hd_map)

    x_clear = []
    y_clear = []
    x_pcd = []
    y_pcd = []

    for i in range(len(pcd_nums)):
            pcd_num = pcd_nums[i]
            current_pcd_path = f"training/PCD_simulated/{fold_num}/{pcd_num}.pcd"
            
            
            
            with open(poses_folder_path) as f:
                json_poses = json.load(f)
                position = json_poses[int(pcd_num)]['position']
            
            clear_points = detector.detect_and_delete(current_pcd_path, position['x'], position['y'])
            
            clear_points[:, 0] -= position['x']
            clear_points[:, 1] -= position['y']
            clear_points[:, 2] -= position['z']
            
            transform = scan_localizer.process_frame(clear_points)

            if i != 0:
                x_clear.append(position['x'])
                y_clear.append(position['y'])
                x_pcd.append(transform[0][3])
                y_pcd.append(transform[1][3])
                
    scan_localizer = Localizer(hd_map)

    x_pcd_deleted = []
    y_pcd_deleted = []

    for i in range(len(pcd_nums)):
        pcd_num = pcd_nums[i]
        current_pcd_path = f"training/PCD_DELETED/{fold_num}/{pcd_num}.pcd"
        
        with open(poses_folder_path) as f:
            json_poses = json.load(f)
            position = json_poses[int(pcd_num)]['position']
        
        clear_points = detector.detect_and_delete(current_pcd_path, position['x'], position['y'])
        
        clear_points[:, 0] -= position['x']
        clear_points[:, 1] -= position['y']
        clear_points[:, 2] -= position['z']
        
        transform = scan_localizer.process_frame(clear_points)

        if i != 0:
            x_pcd_deleted.append(transform[0][3])
            y_pcd_deleted.append(transform[1][3])
            
    x_true_arr = np.array(x_clear)
    y_true_arr = np.array(y_clear)
    x_raw_arr = np.array(x_pcd)
    y_raw_arr = np.array(y_pcd)
    x_filtered_arr = np.array(x_pcd_deleted)
    y_filtered_arr = np.array(y_pcd_deleted)

    error_raw = np.sqrt((x_raw_arr - x_true_arr)**2 + (y_raw_arr - y_true_arr)**2) * 100
    error_filtered = np.sqrt((x_filtered_arr - x_true_arr)**2 + (y_filtered_arr - y_true_arr)**2) * 100

    rmse_raw = np.sqrt(np.mean((error_raw / 100)**2))
    rmse_filtered = np.sqrt(np.mean((error_filtered / 100)**2))

    # print(f"============={fold_num}=============")
    # print("RMSE исходные сканы", rmse_raw, "метров")
    # print("RMSE очищенные сканы", rmse_filtered, "метров")
    # print(f"====================================")
    err1.append(rmse_raw)
    err2.append(rmse_filtered)
    
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


rmse_raw = np.asarray(err1) * 100
rmse_filtered = np.asarray(err2) * 100

locations = np.arange(1, len(rmse_raw) + 1)

plt.figure(figsize=(12, 6))

plt.plot(locations, rmse_raw, color='red', marker='o', markersize=5, linestyle='--', linewidth=1.5, label='Исходные сканы местности', alpha=0.8)
plt.plot(locations, rmse_filtered, color='blue', marker='s', markersize=5, linestyle='-', linewidth=2, label='Очищенные сканы местности')

plt.xlabel('Номер участка местности', fontsize=14)
plt.ylabel('Среднеквадратичная ошибка (сантиметры)', fontsize=14)

plt.legend(loc='upper right', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

plt.xticks(np.arange(1, len(rmse_raw) + 1, step=2))

plt.savefig('rmse_locations_comparison.png', dpi=300, bbox_inches='tight')
plt.show()